# BMIN 5200 — Week 4 in-class exercise
## BFS, DFS, greedy, A*: counting what search costs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week04.ipynb)

**Time:** ~25 minutes · **Pairs with:** Uninformed and heuristic search (state space graphs, BFS/DFS, evaluation functions, greedy best-first, A*)

### What you'll do
- Move a patient from the emergency department to an operating room across a weighted map of a hospital
- Write one `search()` function whose only variable part is how the frontier is ordered, and get five algorithms out of it
- Instrument node expansions and compare cost against work for all five
- Watch greedy take the obvious route and lose, then break A* by handing it an inadmissible heuristic

### Why it matters
Transport, bed assignment, OR scheduling, and differential-diagnosis workups are all search
over a weighted graph, and the thing that separates the algorithms is not cleverness but what
they are allowed to look at: the cost so far, an estimate of the cost remaining, or neither.
The failure you will produce at the end — a heuristic that is confidently wrong and quietly
returns a worse answer while looking faster — is the failure mode of every optimization tool
sold to a health system.

## Setup

Everything used here is preinstalled in Colab. `heapq` is the standard library's priority
queue; it is the only piece of machinery in the whole notebook.

In [ ]:
import heapq
import math

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

print("Setup complete.")

## Part 1 — The map, and one search function

A synthetic floor plan of a hospital. Each unit has an (x, y) position in *minutes of
walking*, and each corridor's traversal time is its straight-line length multiplied by a
congestion factor — most corridors are 1.0, a few are slower. The one corridor to notice is
`south_lobby -> operating_room_3`: geometrically it is the shortest hop on the map, but it
runs through a freight elevator that is shared with materials management, so it takes 13.4
minutes. That single edge is what makes this problem interesting.

We are moving one patient from `emergency_dept` to `operating_room_3`.

In [ ]:
# Positions are in minutes of walking, so straight-line distance is directly comparable
# to traversal time. Synthetic, but the shape is taken from a real transport problem.
UNITS = {
    "emergency_dept":      (0.0, 0.0),
    "ed_imaging":          (1.5, 2.5),
    "radiology_reading":   (3.0, 3.0),
    "central_lab":         (2.5, 1.0),
    "service_corridor":    (4.0, 1.0),
    "south_lobby":         (8.0, 1.0),
    "medical_icu":         (6.0, 2.5),
    "north_corridor":      (2.0, -4.0),
    "elevator_tower_b":    (6.0, -4.0),
    "step_down_unit":      (7.5, -1.0),
    "pre_op_holding":      (9.0, -2.0),
    "operating_room_3":    (10.0, 0.0),
    "main_entrance_lobby": (-5.0, 1.5),
    "outpatient_clinic":   (-8.0, 2.5),
    "registration_desk":   (-6.5, 3.5),
    "loading_dock":        (-6.0, -2.5),
}

# (from, to, congestion factor). 1.0 means you walk it at straight-line speed.
CORRIDORS = [
    ("emergency_dept", "service_corridor", 1.0),
    ("emergency_dept", "north_corridor", 1.0),
    ("emergency_dept", "central_lab", 1.0),
    ("emergency_dept", "ed_imaging", 1.3),
    ("emergency_dept", "main_entrance_lobby", 1.0),
    ("main_entrance_lobby", "outpatient_clinic", 1.0),
    ("main_entrance_lobby", "registration_desk", 1.0),
    ("main_entrance_lobby", "loading_dock", 1.0),
    ("outpatient_clinic", "registration_desk", 1.0),
    ("ed_imaging", "radiology_reading", 1.0),
    ("radiology_reading", "medical_icu", 1.4),
    ("central_lab", "service_corridor", 1.0),
    ("service_corridor", "south_lobby", 1.2),
    ("service_corridor", "medical_icu", 1.0),
    ("medical_icu", "south_lobby", 1.0),
    ("south_lobby", "step_down_unit", 1.0),
    ("south_lobby", "operating_room_3", 6.0),      # shared freight elevator
    ("north_corridor", "elevator_tower_b", 1.0),
    ("elevator_tower_b", "pre_op_holding", 1.0),
    ("elevator_tower_b", "step_down_unit", 1.0),
    ("step_down_unit", "pre_op_holding", 1.0),
    ("pre_op_holding", "operating_room_3", 1.0),
]

START, GOAL = "emergency_dept", "operating_room_3"


def straight_line(unit, other):
    (x1, y1), (x2, y2) = UNITS[unit], UNITS[other]
    return math.hypot(x2 - x1, y2 - y1)


hospital = nx.Graph()
for unit, position in UNITS.items():
    hospital.add_node(unit, position=position)
for here, there, congestion in CORRIDORS:
    hospital.add_edge(here, there, minutes=round(straight_line(here, there) * congestion, 1))

slowest = sorted(hospital.edges(data=True), key=lambda edge: -edge[2]["minutes"])[:5]
print(f"{hospital.number_of_nodes()} units, {hospital.number_of_edges()} corridors\n")
print("the five slowest corridors:")
for here, there, data in slowest:
    print(f"   {here:22s} -> {there:22s} {data['minutes']:5.1f} min")

In [ ]:
SHORT_NAMES = {
    "emergency_dept": "ED", "ed_imaging": "ED imaging", "radiology_reading": "rad reading",
    "central_lab": "lab", "service_corridor": "service corr", "south_lobby": "S lobby",
    "medical_icu": "MICU", "north_corridor": "N corr", "elevator_tower_b": "elev B",
    "step_down_unit": "step-down", "pre_op_holding": "pre-op", "operating_room_3": "OR 3",
    "main_entrance_lobby": "main lobby", "outpatient_clinic": "outpt clinic",
    "registration_desk": "registration", "loading_dock": "dock",
}
positions = nx.get_node_attributes(hospital, "position")

plt.figure(figsize=(11, 6))
nx.draw_networkx(hospital, pos=positions, labels=SHORT_NAMES, node_size=900, font_size=8)
nx.draw_networkx_edge_labels(
    hospital, pos=positions, font_size=7,
    edge_labels={(a, b): data["minutes"] for a, b, data in hospital.edges(data=True)})
plt.title("Hospital transport map: minutes between units (synthetic)")
plt.axis("off")
plt.show()

Here is the entire search algorithm. It keeps a frontier of routes that have been generated
but not yet expanded, pops one, and pushes its neighbors. The **only** thing that will change
between the five algorithms is `frontier_policy`, which returns the sort key for a route: give
it the depth and you get breadth-first, the cost so far and you get uniform-cost, an estimate
of what remains and you get greedy. Read it once, then run breadth-first search.

In [ ]:
def search(graph, start, goal, frontier_policy):
    """Generic graph search. `frontier_policy(unit, minutes_so_far, depth, order)`
    returns the sort key that decides what comes off the frontier next."""
    order = 0                       # generation counter, used to break ties predictably
    frontier = [(frontier_policy(start, 0.0, 0, order), order, start, [start], 0.0)]
    expanded = []                   # the instrumentation: every unit we actually opened

    while frontier:
        _, _, unit, route, minutes = heapq.heappop(frontier)
        if unit in expanded:
            continue                # we already opened this unit by a route we preferred
        expanded.append(unit)

        if unit == goal:
            return {"route": route, "minutes": round(minutes, 1), "stops": len(route) - 1,
                    "expanded": len(expanded), "expansion_order": expanded}

        for neighbor in sorted(graph.neighbors(unit)):
            if neighbor in expanded:
                continue
            order += 1
            step = graph[unit][neighbor]["minutes"]
            heapq.heappush(frontier,
                           (frontier_policy(neighbor, minutes + step, len(route), order),
                            order, neighbor, route + [neighbor], minutes + step))
    return None


def breadth_first(unit, minutes_so_far, depth, order):
    # Shallowest route first; among equally shallow routes, oldest first (a plain queue).
    return (depth, order)


result = search(hospital, START, GOAL, breadth_first)
print("breadth-first")
print("   route   :", " -> ".join(result["route"]))
print("   minutes :", result["minutes"])
print("   stops   :", result["stops"])
print("   expanded:", result["expanded"], "units")

## Part 2 — Four more policies

Breadth-first and depth-first are written for you; they differ by one minus sign, because
depth-first is the same queue read from the other end. The other three are yours. Each is one
line. Right now they are all copies of the breadth-first key, so the cell runs and gives five
identical answers — which is itself the point: these are not five algorithms, they are one
algorithm with five sort keys.

- **uniform-cost**: order by the minutes already spent
- **greedy best-first**: order by the estimated minutes remaining, ignoring what you spent
- **A\***: order by spent plus estimated remaining

In [ ]:
def estimated_remaining(unit):
    """Straight-line minutes from a unit to the OR. Never an overestimate, because
    no corridor is faster than walking in a straight line."""
    return round(straight_line(unit, GOAL), 1)


def depth_first(unit, minutes_so_far, depth, order):
    # Deepest route first; among equally deep routes, newest first (a plain stack).
    return (-depth, -order)


def uniform_cost(unit, minutes_so_far, depth, order):
    # TODO: order by the cost already spent getting here.
    return (depth, order)


def greedy_best_first(unit, minutes_so_far, depth, order):
    # TODO: order by estimated_remaining(unit) alone -- this is the deck's f(n) = h(n).
    return (depth, order)


def a_star(unit, minutes_so_far, depth, order):
    # TODO: order by f(n) = g(n) + h(n): minutes spent plus minutes estimated to remain.
    return (depth, order)


POLICIES = {
    "breadth-first": breadth_first,
    "depth-first": depth_first,
    "uniform-cost": uniform_cost,
    "greedy best-first": greedy_best_first,
    "A* (straight-line h)": a_star,
}

print("estimated minutes remaining, from each unit to OR 3:")
for unit in sorted(UNITS):
    print(f"   {unit:22s} {estimated_remaining(unit):5.1f}")

### Predict before you run

Before you build the comparison table, commit to three answers as a room:

1. Which policies will return the genuinely cheapest route in minutes?
2. Which policy will expand the **fewest** units?
3. Is the answer to (2) the same as the answer to (1)?

Then fill in the three TODOs above, re-run that cell, and run this one.

In [ ]:
comparison = []
for name, policy in POLICIES.items():
    found = search(hospital, START, GOAL, policy)
    comparison.append({
        "algorithm": name,
        "minutes": found["minutes"],
        "stops": found["stops"],
        "units expanded": found["expanded"],
        "route": " -> ".join(SHORT_NAMES[unit] for unit in found["route"]),
    })

print(pd.DataFrame(comparison).to_string(index=False))

Three things to read off that table once your TODOs are in.

**Breadth-first is optimal in the wrong currency.** It returns the route with the fewest
stops — three — and that route goes through the freight elevator and takes about eight
minutes longer than the best one. Fewest hops is only the same as cheapest when every edge
costs the same, which is true in a maze and false in a hospital.

**Greedy is the cheapest search and the worst answer.** It expands four units, fewer than
anything else, because it always walks toward whatever is geometrically nearest the OR. The
south lobby is 2.2 minutes from the OR as the crow flies, so greedy commits to it and inherits
the freight elevator. This is the *Greedy Search: What can go wrong?* slide, with numbers.

**A\* pays a little and gets the optimal route.** It matches uniform-cost's answer while
expanding fewer units, and the ones it skipped are worth naming.

In [ ]:
ucs_order = search(hospital, START, GOAL, uniform_cost)["expansion_order"]
astar_order = search(hospital, START, GOAL, a_star)["expansion_order"]

print("expanded by uniform-cost but NOT by A*:")
for unit in ucs_order:
    if unit not in astar_order:
        print(f"   {unit:22s} (straight-line estimate to OR: {estimated_remaining(unit):.1f} min)")
print()
print("Uniform-cost opens these because they are cheap to reach from the ED. A* refuses,")
print("because cheap-to-reach plus obviously-far-away is still far away.")

### Depth-first has no defense

Depth-first may look respectable in that table. It is not: its answer depends entirely on the
order the corridors happen to be listed in. The only change in the cell below is the tie-break
— which child of the deepest route gets popped first — and it is still, by any definition,
depth-first search.

In [ ]:
def depth_first_other_child(unit, minutes_so_far, depth, order):
    return (-depth, order)      # same algorithm, opposite tie-break


for label, policy in [("depth-first (newest child)", depth_first),
                      ("depth-first (oldest child)", depth_first_other_child)]:
    found = search(hospital, START, GOAL, policy)
    print(f"{label:30s} {found['minutes']:5.1f} min, {found['expanded']:2d} expanded")
    print(f"{'':30s} {' -> '.join(SHORT_NAMES[unit] for unit in found['route'])}")

## Part 3 — Breaking A*

A* is optimal only when the heuristic is **admissible**: it never overestimates the true
remaining cost. Straight-line distance qualifies here by construction, since every corridor's
traversal time is its straight-line length times a factor of at least 1.0.

Now suppose an analyst pulls a year of transport logs, notices that actual transport times
average about five times the straight-line estimate once you count elevator waits and doors,
and "calibrates" the heuristic by scaling it. The estimates are now far closer to reality on
average. Run it, and watch what the scaling factor does to the answer.

In [ ]:
# SCALE is read each time the policy runs, so changing it below changes the heuristic.
SCALE = 1


def a_star_calibrated(unit, minutes_so_far, depth, order):
    return (minutes_so_far + SCALE * estimated_remaining(unit), order)


broken = []
for SCALE in [1, 2, 3, 5]:
    found = search(hospital, START, GOAL, a_star_calibrated)
    broken.append({"heuristic": f"h x {SCALE}", "minutes": found["minutes"],
                   "units expanded": found["expanded"],
                   "route": " -> ".join(SHORT_NAMES[unit] for unit in found["route"])})

greedy = search(hospital, START, GOAL, greedy_best_first)
broken.append({"heuristic": "(greedy, for reference)", "minutes": greedy["minutes"],
               "units expanded": greedy["expanded"],
               "route": " -> ".join(SHORT_NAMES[unit] for unit in greedy["route"])})

print(pd.DataFrame(broken).to_string(index=False))

Every row below the first expands fewer units, and every row below the first is wrong. At
`h x 5` the search has stopped being A* at all: it expands four units, takes the freight
elevator, and returns exactly the route greedy returned — eight minutes worse than optimal on
a patient going to an operating room. If you were benchmarking on speed you would ship it,
because nothing in the output announces that the guarantee is gone.

Scaling a heuristic on purpose is a real technique — weighted A* — and it comes with an
honest bound: the route it returns costs at most `weight` times the optimal. The sin is not
the scaling, it is scaling and still describing the output as the shortest route. To see the
damage directly, compute the true remaining cost from every unit, which is just a
uniform-cost search from that unit to the OR using the function we already have.

In [ ]:
admissibility = []
for unit in sorted(UNITS):
    true_remaining = search(hospital, unit, GOAL, uniform_cost)["minutes"]
    estimate = estimated_remaining(unit)
    admissibility.append({
        "unit": unit,
        "h (straight line)": estimate,
        "h x 5": round(5 * estimate, 1),
        "true remaining": true_remaining,
        "h ok": estimate <= true_remaining,
        "h x 5 ok": 5 * estimate <= true_remaining,
    })

table = pd.DataFrame(admissibility)
print(table.to_string(index=False))
print()
print(f"straight-line heuristic overestimates at {(~table['h ok']).sum()} of {len(table)} units")
print(f"scaled heuristic overestimates at        {(~table['h x 5 ok']).sum()} of {len(table)} units")

One overestimate anywhere on the map is enough. A* stops expanding as soon as it pops the
goal, and an inflated estimate on a unit that lies on the cheapest route makes that route look
worse than it is, so the search commits to something else and never comes back. Admissibility
is not a quality score for the heuristic — it is a precondition, and it either holds or the
answer means nothing.

The practical version of this: any heuristic built by fitting historical data — average
observed transport times, a model of typical delays — is admissible only by accident. The
guarantee comes from a lower bound you can argue for, like "you cannot get there faster than
a straight line," not from a good average.

## Talk about it

1. Greedy expanded four units and got an answer eight minutes worse. For a patient being
   moved to an OR, eight minutes matters; for a scheduler considering ten thousand transports
   overnight, expansion count matters. Where is the crossover, and who in a health system is
   entitled to make that trade?
2. Our heuristic is admissible because it is a physical lower bound. What lower bound could
   you defend for a differential-diagnosis search, where the "distance to the goal" is the
   number and cost of tests still needed to reach a confident diagnosis?
3. The map treats every corridor as available at all times. Real transport has elevators
   taken out of service, contact-precaution routing, and units that stop accepting at shift
   change. Does that change the algorithm, the graph, or the definition of the goal?

## Solutions

Completed versions of the three TODOs, as markdown so nothing runs by accident.

```python
def uniform_cost(unit, minutes_so_far, depth, order):
    return (minutes_so_far, order)


def greedy_best_first(unit, minutes_so_far, depth, order):
    return (estimated_remaining(unit), order)


def a_star(unit, minutes_so_far, depth, order):
    return (minutes_so_far + estimated_remaining(unit), order)
```

Notice what is *not* in any of them. There is no separate implementation of BFS, DFS, UCS,
greedy, or A* — no queue class, no stack, no priority queue variants. All five are the same
loop over the same frontier, and the entire difference between an algorithm that guarantees
the cheapest route and one that does not fits on the right-hand side of a single `return`.

Two details in `search` that are easy to miss and matter:

- `if unit in expanded: continue` is what makes this *graph* search rather than *tree*
  search. Without it the search revisits units and, on a map with cycles like this one, can
  loop indefinitely.
- The `order` counter in every key is not decoration. Without a tie-break, two routes with
  equal priority would be compared on the next element of the tuple, and results would depend
  on dictionary ordering rather than on the algorithm. Reproducibility in a search is a design
  decision, not a property you get for free.